<strong><h1><font color="#2327a5ff">Book Club statistical analysis: web-scraping data from Good Reads</h1></strong>

## Introduction

xxx

## Contents
1. **Data preparation/validation**
   - Load source data
   - Clean and structure the data

2. **Web scraping**
   - Good Reads
   - Other?

3. **Data analysis**
   - Dashboard build

<hr>

## Software versions
This notebook was created in a Jupyer notebook, using the following software versions:
* python 3.12.0
* requests: 2.32.3
* beautifulsoup4: 4.12.3
* pandas: 2.1.4
* textblob: 0.19.0

[TO UPDATE]

## Import libraries and functions

In [ ]:
# Install libraries
#!pip install requests beautifulsoup4 pandas textblob

[TO UPDATE]

In [9]:
# Import libraries
import pandas as pd
pd.set_option('display.max_colwidth', None) # Display full values
import requests
from bs4 import BeautifulSoup
import re
import random
import time

In [10]:
from functions import *

<hr>

## 1. Data preparation / validation

In [11]:
# Read the raw file (skip blank rows)
raw_data = pd.read_csv("C:\\Users\\michael.aspinall\\APPRENTICESHIP DEGREE\\Portfolio\\book_club.csv", header=None)
goodreads_links = pd.read_csv("C:\\Users\\michael.aspinall\\APPRENTICESHIP DEGREE\\Portfolio\\goodreads_links.csv")

In [12]:
ratings = parse_stacked_ratings(raw_data)

In [13]:
# Drop rows with missing Total score
ratings = ratings[(~ratings["Score"].isna()) &
                  (ratings["Score"] != None) & 
                  (ratings["Score"] != "None") &
                  (ratings["Score"] != "#DIV/0!") &
                  (ratings["Score"] != "0") &
                  (ratings["Score"] != 0)
                  ]

In [14]:
print(ratings.head(10))

                Book Chooser              Criterion Reviewer Score
0   The Night Circus   Clare                   Plot     Mike   7.0
1   The Night Circus   Clare                   Plot      Nia   6.7
2   The Night Circus   Clare                   Plot     Kath   5.0
3   The Night Circus   Clare                   Plot      Jen   7.0
7   The Night Circus   Clare     Quality of writing     Mike   9.0
8   The Night Circus   Clare     Quality of writing      Nia   8.9
9   The Night Circus   Clare     Quality of writing     Kath   6.0
10  The Night Circus   Clare     Quality of writing      Jen   8.5
14  The Night Circus   Clare  Character development     Mike   5.0
15  The Night Circus   Clare  Character development      Nia   4.2


In [15]:
# Change datatypes
ratings['Score'] = pd.to_numeric(ratings['Score'], downcast='float')

In [16]:
print(ratings['Score'].sum())

6139.275


In [17]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 900 entries, 0 to 1181
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Book       900 non-null    object 
 1   Chooser    900 non-null    object 
 2   Criterion  900 non-null    object 
 3   Reviewer   900 non-null    object 
 4   Score      900 non-null    float32
dtypes: float32(1), object(4)
memory usage: 38.7+ KB


In [18]:
display(ratings[ratings['Score'] == 1])

,Book,Chooser,Criterion,Reviewer,Score
483,The Dice Man,Mike,Overall experience / enjoyment,Nia,1.0
547,A Visit From The Goon Squad,Clare,Plot,Kath,1.0
556,A Visit From The Goon Squad,Clare,Quality of writing,Kath,1.0
565,A Visit From The Goon Squad,Clare,Character development,Kath,1.0
574,A Visit From The Goon Squad,Clare,Overall experience / enjoyment,Kath,1.0
577,A Visit From The Goon Squad,Clare,Overall experience / enjoyment,Gareth,1.0
583,A Visit From The Goon Squad,Clare,Total,Kath,1.0
635,The Nicander Chronicles,Dunc,Plot,Mike,1.0
636,The Nicander Chronicles,Dunc,Plot,Nia,1.0
653,The Nicander Chronicles,Dunc,Character development,Mike,1.0


In [19]:
# Tidy up names (Dunc/Duncan, Fred/Freddie, etc)
names_dict = {'Fred': 'Freddie',
              'Dunc': 'Duncan',
              'Katherine': 'Kath',
              'Jennifer': 'Jen'}

ratings['Chooser'] = ratings['Chooser'].replace(names_dict)
ratings['Reviewer'] = ratings['Reviewer'].replace(names_dict)


### Merge in goodreads_links

In [20]:
ratings = ratings.merge(goodreads_links, on='Book', how='left')

In [21]:
ratings.sample(5)

,Book,Chooser,Criterion,Reviewer,Score,goodreads_url
568,Conclave,Freddie,Overall experience / enjoyment,Clare,7.00,https://www.goodreads.com/book/show/29397486-conclave
869,Demon Copperfield,Clare,Total,Freddie,8.75,NaN
354,A Passage To India,Kath,Overall experience / enjoyment,Nia,6.50,https://www.goodreads.com/book/show/45195.A_Passage_to_India
266,The Wonderful Story of Henry Sugar and Six More,Gareth,Plot,Nia,6.00,https://www.goodreads.com/book/show/6671.The_Wonderful_Story_of_Henry_Sugar_and_Six_More
773,The House of Doors,Nia,Total,Jen,5.75,https://www.goodreads.com/book/show/65215270-the-house-of-doors


## 2. EDA

In [22]:
# Avg of reviews of someone's own book choice
self_review_avg = ratings['Score'][(ratings['Criterion'] == 'Total') &
                         (ratings['Chooser'] == ratings['Reviewer'])
                         ].mean()

# Avg reviews overall
review_avg = ratings['Score'][(ratings['Criterion'] == 'Total')].mean()


print(f"Avg total review score when someone reviews their own choice: {self_review_avg:.2f}")
print(f"Avg total review score overall: {review_avg:.2f}")

print(f"-> People reviewed their own books ~{(((self_review_avg/review_avg)*100)-100):.2f}% ({self_review_avg-review_avg:.2f}ppt) more favourably")

Avg total review score when someone reviews their own choice: 7.42
Avg total review score overall: 6.82
-> People reviewed their own books ~8.72% (0.59ppt) more favourably


## 3. Web scraping

### Scrape Circe and save soup for future dev

In [23]:
# Scrape genre
#soup = scrape_website('https://www.goodreads.com/book/show/35959740-circe')
#soup.prettify

In [24]:
#with open("circe.html", "w", encoding="utf-8") as f:
 #   f.write(soup.prettify())

### Scrape multiple books at once

In [25]:
# Create dictionary of books and URLs to scrape

books_urls = ratings[['Book', 'goodreads_url']].drop_duplicates()

# Convert to dictionary
book_dict = dict(books_urls.values)
book_dict

{'The Night Circus': 'https://www.goodreads.com/book/show/9361589-the-night-circus',
 'Ask The Dust': 'https://www.goodreads.com/book/show/46227.Ask_the_Dust',
 'I Know Why The Caged Bird Sings': 'https://www.goodreads.com/book/show/13214.I_Know_Why_the_Caged_Bird_Sings',
 'Circe': 'https://www.goodreads.com/book/show/35959740-circe',
 'The Beekeeper of Aleppo': 'https://www.goodreads.com/book/show/43124137-the-beekeeper-of-aleppo',
 "All That's Left In The World": 'https://www.goodreads.com/book/show/58329296-all-that-s-left-in-the-world',
 'Apeirogon by Colum McCann': 'https://www.goodreads.com/book/show/50024187-apeirogon',
 'The Offing': 'https://www.goodreads.com/book/show/43412959-the-offing',
 'Happiness': 'https://www.goodreads.com/book/show/35458040-happiness',
 'The Wonderful Story of Henry Sugar and Six More': 'https://www.goodreads.com/book/show/6671.The_Wonderful_Story_of_Henry_Sugar_and_Six_More',
 'Crying in H Mart': 'https://www.goodreads.com/book/show/54814676-crying-i

In [ ]:
goodreads_data = pd.DataFrame()

for key, value in book_dict.items():
    print(f'Scraping {key}...')
    df = scrape_details(key,value)

    # Combine all books into one master df
    goodreads_data = pd.concat([goodreads_data, df])
    
    print(f"\t{key} details scraped.")
    print('-' * 25)
            
    # Add a random delay to mimic human behavior
    delay = random.uniform(5, 10)
    print(f"Waiting {delay:.2f} seconds before next request...")
    time.sleep(delay)
    

In [26]:
master_data = pd.DataFrame()

for key, value in book_dict.items():
    print(f'Scraping {key}...')
    df = scrape_details(key,value)

    # Combine all books into one master df
    master_data = pd.concat([master_data, df])
    
    print(f"\t{key} details scraped.")
            
    # Add a random delay to mimic human behavior
    delay = random.uniform(5, 10)
    print(f"\nWaiting {delay:.2f} seconds before next request...\n")
    time.sleep(delay)
    

Scraping The Night Circus...


	The Night Circus details scraped.

Waiting 7.58 seconds before next request...

Scraping Ask The Dust...
	Ask The Dust details scraped.

Waiting 8.19 seconds before next request...

Scraping I Know Why The Caged Bird Sings...
	I Know Why The Caged Bird Sings details scraped.

Waiting 6.56 seconds before next request...

Scraping Circe...
	Circe details scraped.

Waiting 8.92 seconds before next request...

Scraping The Beekeeper of Aleppo...
	The Beekeeper of Aleppo details scraped.

Waiting 9.17 seconds before next request...

Scraping All That's Left In The World...
	All That's Left In The World details scraped.

Waiting 7.66 seconds before next request...

Scraping Apeirogon by Colum McCann...
	Apeirogon by Colum McCann details scraped.

Waiting 9.94 seconds before next request...

Scraping The Offing...
	The Offing details scraped.

Waiting 6.05 seconds before next request...

Scraping Happiness...
	Happiness details scraped.

Waiting 7.62 seconds before next request...

Scraping 

MissingSchema: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?

In [ ]:
master_data.head()

,Book,image,author_link,author_name,genres,description,goodreads_rating,no_ratings,no_reviews,publication_date
0,The Night Circus,https://images-na.ssl-images-amazon.com/images/S/compressed.photo.goodreads.com/books/1387124618i/9361589.jpg,https://www.goodreads.com/author/show/4370565.Erin_Morgenstern,Erin Morgenstern,"[Fantasy, Fiction, Romance, Historical Fiction, Magical Realism, Magic]","The circus arrives without warning. No announcements precede it. It is simply there, when yesterday it was not. Within the black-and-white striped canvas tents is an utterly unique experience full of breathtaking amazements. It is called Le Cirque des Rêves, and it is only open at night. But behind the scenes, a fierce competition is underway—a duel between two young magicians, Celia and Marco, who have been trained since childhood expressly for this purpose by their mercurial instructors. Unbeknownst to them, this is a game in which only one can be left standing, and the circus is but the stage for a remarkable battle of imagination and will. Despite themselves, however, Celia and Marco tumble headfirst into love—a deep, magical love that makes the lights flicker and the room grow warm whenever they so much as brush hands. True love or not, the game must play out, and the fates of everyone involved, from the cast of extraordinary circus performers to the patrons, hang in the balance, suspended as precariously as the daring acrobats overhead. Written in rich, seductive prose, this spell-casting novel is a feast for the senses and the heart.",4.00,1076834,115370,"September 13, 2011"
0,Ask The Dust,https://images-na.ssl-images-amazon.com/images/S/compressed.photo.goodreads.com/books/1388283697i/46227.jpg,https://www.goodreads.com/author/show/25864.John_Fante,John Fante,"[Fiction, Classics, American, Literature, Romance]","""Fante was my god."" —Charles Bukowski, in his introduction to Ask the DustArturo Bandini, a young, struggling Italian-American writer living in a seedy hotel in 1930s Los Angeles, falls hard for the elusive, mocking, unstable Camilla Lopez, a Mexican waitress. The pair embark on a strange and strained love-hate relationship, which slowly, but inexorably, descends into the realm of madness.Ask the Dust is one of the truly great, yet unsung, American novels of the twentieth century. A tough and unsentimental story with a soft and tender heart, it remains as fresh and affecting as the day it was written.",4.10,38640,2941,"January 1, 1939"
0,I Know Why The Caged Bird Sings,https://images-na.ssl-images-amazon.com/images/S/compressed.photo.goodreads.com/books/1327957927i/13214.jpg,https://www.goodreads.com/author/show/3503.Maya_Angelou,Maya Angelou,"[Nonfiction, Classics, Memoir, Biography, Autobiography, Poetry, Feminism]","Maya Angelou’s debut memoir is a modern American classic beloved worldwide. Her life story is told in the documentary film And Still I Rise, as seen on PBS’s American Masters.Here is a book as joyous and painful, as mysterious and memorable, as childhood itself. I Know Why the Caged Bird Sings captures the longing of lonely children, the brute insult of bigotry, and the wonder of words that can make the world right. Maya Angelou’s debut memoir is a modern American classic beloved worldwide. Sent by their mother to live with their devout, self-sufficient grandmother in a small Southern town, Maya and her brother, Bailey, endure the ache of abandonment and the prejudice of the local “powhitetrash.” At eight years old and back at her mother’s side in St. Louis, Maya is attacked by a man many times her age—and has to live with the consequences for a lifetime. Years later, in San Francisco, Maya learns that love for herself, the kindness of others, her own strong spirit, and the ideas of great authors (“I met and fell in love with William Shakespeare”) will allow her to be free instead of imprisoned. Poetic and powerful, I Know Why the Caged Bird Sings will touch hearts and change minds for as long as people rea

### Merge scraped data and Book Club data

In [27]:
master_data = pd.merge(ratings, master_data, on='Book', how='left')

In [30]:
master_data.sample()

,Book,Chooser,Criterion,Reviewer,Score,goodreads_url,image,author_link,author_name,genres,description,goodreads_rating,no_ratings,no_reviews,publication_date
152,All That's Left In The World,Mike,Quality of writing,Gareth,6.0,https://www.goodreads.com/book/show/58329296-all-that-s-left-in-the-world,https://images-na.ssl-images-amazon.com/images/S/compressed.photo.goodreads.com/books/1624548105i/58329296.jpg,https://www.goodreads.com/author/show/21192142.Erik_J_Brown,Erik J. Brown,"[Romance, LGBT, Young Adult, Queer, Dystopia, Science Fiction, Fiction]","What If It's Us meets They Both Die at the End in this postapocalyptic, queer YA adventure romance from debut author Erik J. Brown. Perfect for fans of Adam Silvera, Alex London, and Heartstopper by Alice Oseman.When Andrew stumbles upon Jamie's house, he's injured, starved, and has nothing left to lose. A deadly pathogen has killed off most of the world's population, including everyone both boys have ever loved. And if this new world has taught them anything, it's to be scared of what other desperate people will do . . . so why does it seem so easy for them to trust each other?After danger breaches their shelter, they flee south in search of civilization. But something isn't adding up about Andrew's story, and it could cost them everything. And Jamie has a secret, too. He's starting to feel something more than friendship for Andrew, adding another layer of fear and confusion to an already tumultuous journey.The road ahead of them is long, and to survive, they'll have to shed their secrets, face the consequences of their actions, and find the courage to fight for the future they desire, together. Only one thing feels certain: all that's left in their world is the undeniable pull they have toward each other.",4.22,40412,7080,"March 8, 2022"


In [31]:
### Save dataframe
master_data.to_csv('master_data.csv', sep=',', index=False, encoding='utf-8')
print(f"Data saved to filename 'master_data.csv' in current directory")

Data saved to filename 'master_data.csv' in current directory
